<a href="https://colab.research.google.com/github/adautojr001/AprendizadodeMaquina-Adauto/blob/master/alerta_tinyML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Alerta criado para microcontroladores (TinyML)
import sqlite3
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from google.colab import files

# Upload do banco de dados
uploaded = files.upload()  # selecione 'iot_agro.db' do seu PC

# Carregar dados do banco
conn = sqlite3.connect("iot_agro.db")
cursor = conn.cursor()
cursor.execute("SELECT temperatura, umidade, luminosidade, red, nir, ndvi FROM sensores")
dados = np.array(cursor.fetchall(), dtype=np.float32)
conn.close()

# Criar rótulos simples (alerta)
def criar_rotulo(linha):
    temperatura, umidade, luminosidade, red, nir, ndvi = linha
    if umidade < 40 or temperatura > 30 or (ndvi is not None and ndvi < 0.2):
        return 1
    return 0

y = np.array([criar_rotulo(l) for l in dados], dtype=np.float32)

# Normalizar e tratar NaN
dados = np.nan_to_num(dados, nan=0.0)
dados = (dados - np.mean(dados, axis=0)) / np.std(dados, axis=0)

# Separar treino e teste
X_train, X_test, y_train, y_test = train_test_split(dados, y, test_size=0.2, random_state=42)

# Construir modelo pequeno
modelo = keras.Sequential([
    keras.layers.Dense(8, activation='relu', input_shape=(6,)),
    keras.layers.Dense(4, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Treinar modelo
modelo.fit(X_train, y_train, epochs=50, batch_size=8, verbose=1, validation_data=(X_test, y_test))

# Converter para TinyML (.tflite) com otimização
converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantização para MCU
modelo_tflite = converter.convert()

# Salvar modelo
with open("modelo_alerta.tflite", "wb") as f:
    f.write(modelo_tflite)

print("Modelo TinyML salvo como modelo_alerta.tflite")


Saving iot_agro.db to iot_agro (3).db
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


564/564 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7428 - loss: 0.5620 - val_accuracy: 0.9025 - val_loss: 0.2905
Epoch 2/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8864 - loss: 0.2725 - val_accuracy: 0.9459 - val_loss: 0.1514
Epoch 3/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9404 - loss: 0.1576 - val_accuracy: 0.9707 - val_loss: 0.1015
Epoch 4/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9660 - loss: 0.1077 - val_accuracy: 0.9761 - val_loss: 0.0818
Epoch 5/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9684 - loss: 0.0927 - val_accuracy: 0.9761 - val_loss: 0.0735
Epoch 6/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9729 - loss: 0.0799 - val_accuracy: 0.9770 - val_loss: 0.0660
Epoch 7/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9735 - loss: 0.0747 - val_accuracy: 0.9787 - val_loss: 0.0609
Epoch 8/50
564/564 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9732 - loss: 0.0691 - val_accuracy: 0.9849 - val_